# Training Registry Usage Demo

This notebook demonstrates how to use the training data registry for different ML tasks.

## What is the Registry?

The registry is a **unified data pointer system** that tracks:
- Protein structures (one per UniProt)
- Active compounds with train/test splits at multiple similarity thresholds
- Decoy compounds for each protein
- Paths to all raw data files

**Key advantage**: Lazy loading - features computed on-demand, not pre-computed

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

## 1. Load and Explore Registry

In [ ]:
# Load registry
registry = pd.read_csv('../../training_data/registry.csv')

print(f"Total samples: {len(registry):,}")
print(f"\nColumns: {list(registry.columns)}")
print(f"\nFirst 3 rows:")
registry.head(3)

In [ ]:
# Check split distribution
print("Split Distribution:")
print(registry['split'].value_counts())
print(f"\nActive vs Decoy:")
print(registry['is_active'].value_counts())

In [ ]:
# Check similarity thresholds (actives only)
actives = registry[registry['is_active'] == True]
print("Similarity Threshold Distribution:")
print(actives['similarity_threshold'].value_counts())

# UniProts
print(f"\nNumber of UniProts: {registry['uniprot_id'].nunique()}")
print(f"\nTop 5 UniProts by sample count:")
print(registry['uniprot_id'].value_counts().head())

## 2. Understanding the Split Logic

**Important**: Test set = ALL actives - train set for that threshold

Let's verify this for one UniProt:

In [ ]:
# Pick one UniProt
uniprot = 'O00141'
threshold = '0p7'

# Get all samples for this UniProt
uniprot_data = registry[registry['uniprot_id'] == uniprot]

# Get train and test sets for 0p7 threshold
train_0p7 = uniprot_data[
    (uniprot_data['split'] == 'train') & 
    (uniprot_data['similarity_threshold'] == threshold)
]
test_0p7 = uniprot_data[
    (uniprot_data['split'] == 'test') & 
    (uniprot_data['similarity_threshold'] == threshold)
]
decoys = uniprot_data[uniprot_data['split'] == 'decoy']

print(f"UniProt {uniprot} at threshold {threshold}:")
print(f"  Train: {len(train_0p7)} compounds")
print(f"  Test:  {len(test_0p7)} compounds")
print(f"  Total actives (train + test): {len(train_0p7) + len(test_0p7)}")
print(f"  Decoys: {len(decoys)} compounds")

# Verify no overlap
train_smiles = set(train_0p7['smiles'])
test_smiles = set(test_0p7['smiles'])
overlap = train_smiles.intersection(test_smiles)
print(f"\n✓ Train/Test overlap: {len(overlap)} (should be 0)")

## 3. Use Case: Binary Classification (Active vs Decoy)

Train model to distinguish active compounds from decoys

In [ ]:
# Get training data: actives + decoys at 0p7 threshold
train_actives = registry[
    (registry['split'] == 'train') & 
    (registry['similarity_threshold'] == '0p7')
]
train_decoys = registry[registry['split'] == 'decoy']

train_set = pd.concat([train_actives, train_decoys])

print(f"Training set:")
print(f"  Actives: {len(train_actives):,}")
print(f"  Decoys:  {len(train_decoys):,}")
print(f"  Total:   {len(train_set):,}")
print(f"  Class balance: {len(train_actives)/len(train_set)*100:.1f}% active")

# Test set: only actives (no decoys in test)
test_actives = registry[
    (registry['split'] == 'test') & 
    (registry['similarity_threshold'] == '0p7')
]

print(f"\nTest set:")
print(f"  Actives: {len(test_actives):,}")

### Compute Fingerprints for Training

In [ ]:
def compute_fingerprint(smiles, radius=2, n_bits=2048):
    """Compute Morgan fingerprint from SMILES."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
        return np.array(fp, dtype=np.float32)
    except:
        return None

# Compute fingerprints for a few samples
sample_size = 100
samples = train_set.sample(min(sample_size, len(train_set)))

print(f"Computing fingerprints for {len(samples)} samples...")
fingerprints = []
labels = []

for idx, row in samples.iterrows():
    fp = compute_fingerprint(row['smiles'])
    if fp is not None:
        fingerprints.append(fp)
        labels.append(1 if row['is_active'] else 0)

X = np.array(fingerprints)
y = np.array(labels)

print(f"\nFeature matrix shape: {X.shape}")
print(f"Labels: {np.sum(y)} actives, {len(y) - np.sum(y)} decoys")
print(f"Fingerprint density: {X.mean():.3f}")

## 4. Use Case: Regression (Predict Affinity)

Note: Currently affinity values are not populated, but here's how you would use them:

In [ ]:
# Check how many samples have affinity data
has_affinity = registry['affinity_value'].notna()
print(f"Samples with affinity: {has_affinity.sum():,} / {len(registry):,} ({has_affinity.sum()/len(registry)*100:.1f}%)")

# When affinity data is available, use it like this:
if has_affinity.sum() > 0:
    with_affinity = registry[has_affinity]
    print(f"\nAffinity statistics:")
    print(with_affinity['affinity_value'].describe())
    
    # Convert to pIC50 (typically used in ML)
    # pIC50 = -log10(IC50 in M)
    # If IC50 is in nM: pIC50 = -log10(IC50_nM * 1e-9)
    with_affinity['pIC50'] = -np.log10(with_affinity['affinity_value'] * 1e-9)
    print(f"\npIC50 statistics:")
    print(with_affinity['pIC50'].describe())
else:
    print("\n⚠️  No affinity data currently populated in registry")
    print("    Affinity matching can be added by updating the registry builder")

## 5. Access Raw Data Files

In [ ]:
# Get a sample with 3D structure
sample_with_sdf = registry[registry['sdf_path'].notna()].iloc[0]

print("Sample with 3D structure:")
print(f"  UniProt: {sample_with_sdf['uniprot_id']}")
print(f"  PDB: {sample_with_sdf['pdb_id']}")
print(f"  SMILES: {sample_with_sdf['smiles'][:60]}...")
print(f"  Protein structure: {sample_with_sdf['cif_path']}")
print(f"  Ligand 3D: {sample_with_sdf['sdf_path']}")
print(f"  Resolution: {sample_with_sdf['resolution']} Å")
print(f"  Quality score: {sample_with_sdf['quality_score']}")

In [ ]:
# Load 3D structure from SDF
if pd.notna(sample_with_sdf['sdf_path']):
    try:
        suppl = Chem.SDMolSupplier(sample_with_sdf['sdf_path'], sanitize=False, removeHs=False)
        mol_3d = next(suppl)
        
        if mol_3d:
            Chem.SanitizeMol(mol_3d)
            print(f"\n✓ Loaded 3D structure:")
            print(f"  Atoms: {mol_3d.GetNumAtoms()}")
            print(f"  Bonds: {mol_3d.GetNumBonds()}")
            
            # Check if has 3D coordinates
            if mol_3d.GetNumConformers() > 0:
                conf = mol_3d.GetConformer()
                print(f"  Has 3D coordinates: Yes")
                # Get first atom position
                pos = conf.GetAtomPosition(0)
                print(f"  Example coords: ({pos.x:.2f}, {pos.y:.2f}, {pos.z:.2f})")
    except Exception as e:
        print(f"Error loading SDF: {e}")

## 6. Molecular Property Analysis

In [ ]:
def compute_properties(smiles):
    """Compute molecular properties."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return {
            'mw': Descriptors.MolWt(mol),
            'logp': Descriptors.MolLogP(mol),
            'hbd': Descriptors.NumHDonors(mol),
            'hba': Descriptors.NumHAcceptors(mol),
            'tpsa': Descriptors.TPSA(mol),
            'rotatable': Descriptors.NumRotatableBonds(mol)
        }
    except:
        return None

# Compute for a subset (actives vs decoys)
n_samples = 200
actives_sample = registry[registry['is_active'] == True].sample(min(n_samples, len(registry[registry['is_active'] == True])))
decoys_sample = registry[registry['is_active'] == False].sample(min(n_samples, len(registry[registry['is_active'] == False])))

print("Computing properties...")
active_props = [compute_properties(s) for s in actives_sample['smiles']]
active_props = [p for p in active_props if p is not None]

decoy_props = [compute_properties(s) for s in decoys_sample['smiles']]
decoy_props = [p for p in decoy_props if p is not None]

print(f"Computed properties for {len(active_props)} actives and {len(decoy_props)} decoys")

In [ ]:
# Plot property distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Molecular Properties: Actives vs Decoys', fontsize=16)

properties = ['mw', 'logp', 'hbd', 'hba', 'tpsa', 'rotatable']
titles = ['Molecular Weight', 'LogP', 'H-Bond Donors', 'H-Bond Acceptors', 'TPSA', 'Rotatable Bonds']

for idx, (prop, title) in enumerate(zip(properties, titles)):
    ax = axes[idx // 3, idx % 3]
    
    active_vals = [p[prop] for p in active_props]
    decoy_vals = [p[prop] for p in decoy_props]
    
    ax.hist(active_vals, bins=30, alpha=0.6, label='Actives', color='blue')
    ax.hist(decoy_vals, bins=30, alpha=0.6, label='Decoys', color='red')
    ax.set_xlabel(title)
    ax.set_ylabel('Count')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nProperty Statistics:")
print("\nActives:")
for prop, title in zip(properties, titles):
    vals = [p[prop] for p in active_props]
    print(f"  {title:20s}: {np.mean(vals):6.2f} ± {np.std(vals):6.2f}")

print("\nDecoys:")
for prop, title in zip(properties, titles):
    vals = [p[prop] for p in decoy_props]
    print(f"  {title:20s}: {np.mean(vals):6.2f} ± {np.std(vals):6.2f}")

## 7. Practical Example: Create Training Batches

In [ ]:
def create_batch(registry_df, batch_size=32, include_protein_features=False):
    """
    Create a training batch from registry.
    
    Args:
        registry_df: Filtered registry DataFrame
        batch_size: Number of samples per batch
        include_protein_features: Whether to include protein info
    
    Returns:
        Dictionary with features and labels
    """
    batch_samples = registry_df.sample(min(batch_size, len(registry_df)))
    
    # Compute fingerprints
    fingerprints = []
    labels = []
    sample_ids = []
    
    for idx, row in batch_samples.iterrows():
        fp = compute_fingerprint(row['smiles'])
        if fp is not None:
            fingerprints.append(fp)
            labels.append(1 if row['is_active'] else 0)
            sample_ids.append(row['sample_id'])
    
    batch = {
        'fingerprints': np.array(fingerprints),
        'labels': np.array(labels),
        'sample_ids': sample_ids
    }
    
    if include_protein_features:
        batch['uniprot_ids'] = batch_samples['uniprot_id'].tolist()
        batch['pdb_ids'] = batch_samples['pdb_id'].tolist()
        batch['resolutions'] = batch_samples['resolution'].tolist()
    
    return batch

# Example: Create a training batch
print("Creating training batch...")
batch = create_batch(train_set, batch_size=32, include_protein_features=True)

print(f"\nBatch contents:")
print(f"  Fingerprints shape: {batch['fingerprints'].shape}")
print(f"  Labels shape: {batch['labels'].shape}")
print(f"  Labels: {batch['labels']}")
print(f"  Sample IDs: {batch['sample_ids'][:3]}...")
print(f"  UniProt IDs: {set(batch['uniprot_ids'])}")

## Summary

The registry provides:

1. **Unified data access** - Single CSV with all data pointers
2. **Flexible splits** - 3 similarity thresholds (0p3, 0p5, 0p7)
3. **Lazy loading** - Compute features on-demand
4. **Multiple tasks** - Classification (active/decoy), regression (affinity)
5. **Raw data access** - Protein structures (CIF), ligand 3D (SDF), SMILES

### Next Steps:
- Use with the provided dataloaders in `dataloaders.py`
- Train RF/XGBoost, GNN, or transformer models
- Add custom feature extraction as needed
- Implement your own dataloader for specific architectures